### 1. What is a Tool?

A tool is simply a function that the AI can call to perform an action or get information.

In [ ]:
# For example, we can create a weather tool:

from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get the weather for a city."""
    return f"The weather in {city} is sunny."



# Here:

@tool

# tells LangChain:
# "Make this Python function available as an AI tool."



# The function itself is still just Python code:

def get_weather(city: str) -> str:
    return f"The weather in {city} is sunny."

# The tool has:

# Name: get_weather
# Input: city
# Output: str
# Description: "Get the weather for a city."





# Calling the tool yourself
# You can call it directly:

result = get_weather.invoke({
    "city": "Bhubaneswar"
})

print(result)

# Output:

# The weather in Bhubaneswar is sunny.
# At this point, Gemini has not done anything.
# You manually called the tool.




# So:

# Your Python code
#       ↓
# get_weather()
#       ↓
# Result

# That's simply a tool/function call.

### 2. Why do we need Tools?

In [ ]:
# An LLM normally generates text.

# For example:

# User
#  ↓
# Gemini
#  ↓
# Text response

# But an application often needs the AI to do something outside the model.



# For example:

# Get weather
# Search Google
# Query database
# Calculate something
# Read a file
# Send an email
# Call an API
# Create an appointment
# Execute application logic


# The LLM itself doesn't automatically perform these operations.



# Tools provide that connection:

#                  ┌── Weather API
#                  ├── Database
# Gemini ── Tools ─┼── Calculator
#                  ├── Search
#                  └── Your Python functions



# So you can think of a tool as a bridge between the LLM and your application/external systems.

### 3. Tool vs Agent

In [ ]:
# This distinction is very important.




# Tool -


# A tool is something the model can use.
# ( get_weather tool )

# But the tool itself doesn't decide when to use itself.

# For example:

# get_weather.invoke({
#     "city": "Bhubaneswar"
# })

# You explicitly called it.





# Agent

# An agent can decide which tool to use and when to use it.

# For example:

# User:
# What's the weather in Bhubaneswar?
#         ↓
#        Gemini
#         ↓
# "I need weather information."
#         ↓
#    get_weather
#         ↓
# Tool result
#         ↓
#        Gemini
#         ↓
# Final answer




# So remember:

# Tool = capability
# Agent = decision-making + tool usage

### 4. Give the Tool to Gemini

In [3]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)


@tool
def get_weather(city: str) -> str:
    """Get the weather for a city."""
    return f"The weather in {city} is sunny."


tools = [get_weather]

llm_with_tools = llm.bind_tools(tools)


# The important line is:

# llm_with_tools = llm.bind_tools(tools)

# It tells Gemini: "These tools are available to you."
# It does not mean that the tool has already been executed.


response = llm_with_tools.invoke(
    "What's the weather in Bhubaneswar?"
)

print(response)

content=[] additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"city": "Bhubaneswar"}'}, '__gemini_function_call_thought_signatures__': {'call_1543960': 'El4KXAFpFH0TJrEnhN//btV7HSx7TAjemoirfXWSL6inaa/cHSEGr2o6P8szegrrE9pP/ivOZmq0XZ92rRwD7+7yVo9C66eT5H0WTjLU48Ktjrf5RtdCpGCoTiqAPi/u'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a0baa4-4d46-78c1-a806-9e0600a22906-0' tool_calls=[{'name': 'get_weather', 'args': {'city': 'Bhubaneswar'}, 'id': 'call_1543960', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 53, 'output_tokens': 18, 'total_tokens': 71, 'input_token_details': {'cache_read': 0}}


In [ ]:
# Now:

response = llm_with_tools.invoke(
    "What's the weather in Bhubaneswar?"
)

print(response)

# Gemini may decide that it needs the weather tool.

# Instead of immediately giving a normal text answer, the response can contain a tool call.

# Inspect it:

print(response.tool_calls)

# Conceptually, you may get:

[
    {
        "name": "get_weather",
        "args": {
            "city": "Bhubaneswar"
        }
    }
]

# This means:

# Gemini is requesting your application to call get_weather with "Bhubaneswar".

In [ ]:
# This is one of the most important concepts.

# When Gemini produces:

{
    "name": "get_weather",
    "args": {
        "city": "Bhubaneswar"
    }
}

# Gemini has only requested the tool call.

# Your Python application still needs to execute:

get_weather.invoke({
    "city": "Bhubaneswar"
})

# So there are two separate steps:

# 1. Gemini decides:
#    "Call get_weather with Bhubaneswar"

# 2. Your application:
#    Actually executes get_weather()

# This distinction becomes extremely important when you learn agents.

### 5. Complete Manual Tool-Calling Example

In [4]:
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool

load_dotenv()


# 1. Create the tool
@tool
def get_weather(city: str) -> str:
    """Get the weather for a city."""
    return f"The weather in {city} is sunny."


# 2. Give the tool to Gemini
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

llm_with_tools = llm.bind_tools([get_weather])


# 3. Ask Gemini
response = llm_with_tools.invoke(
    "What's the weather in Bhubaneswar?"
)


# 4. See what tool Gemini requested
print(response.tool_calls)

# Now the application can exicute this
tool_call = response.tool_calls[0]

tool_result = get_weather.invoke(
    tool_call["args"]
)

print(tool_result)


# So now the complete flow is:

# User
#  ↓
# Gemini
#  ↓
# Tool call request
#  ↓
# Your Python code
#  ↓
# get_weather()
#  ↓
# Tool result

[{'name': 'get_weather', 'args': {'city': 'Bhubaneswar'}, 'id': 'call_1058236', 'type': 'tool_call'}]
The weather in Bhubaneswar is sunny.


### 6. But Gemini still needs the Tool Result

In [ ]:
# There is one more important step.



# Suppose Gemini requested:

# get_weather("Bhubaneswar")



# and the tool returned:

# The weather in Bhubaneswar is sunny.


# Gemini needs to see that result before it can produce the final natural-language response.




# Conceptually:

# User
#  ↓
# Gemini
#  ↓
# Tool call
#  ↓
# get_weather()
#  ↓
# Tool result
#  ↓
# Gemini
#  ↓
# Final answer

# This is why tool calling is naturally a loop.

### 7. Multiple Tools

In real applications, you'll usually have multiple tools.

In [ ]:
from langchain_core.tools import tool


@tool
def get_weather(city: str) -> str: 
    """Get the weather for a city."""   # Tool description (Important to have to get tool info)
    return f"The weather in {city} is sunny."


@tool
def get_population(city: str) -> str:
    """Get the population of a city."""  # Tool description (Important to have to get tool info)
    return f"The population of {city} is approximately 1 million."


tools = [
    get_weather,
    get_population
]

llm_with_tools = llm.bind_tools(tools)

In [ ]:
# Now Gemini has access to:

#              Gemini
#                 │
#         ┌───────┴────────┐
#         ↓                ↓
#  get_weather       get_population




# For:

# What's the weather in Bhubaneswar?

# Gemini can request:

# get_weather(
#     city="Bhubaneswar"
# )




# For:

# What's the population of Bhubaneswar?

# it can request:

# get_population(
#     city="Bhubaneswar"
# )



# The model chooses the appropriate tool based on the available tool 
# descriptions and the user's request.

### 8. Tool Calling vs Agent

In [ ]:
# TOOL
# ↓
# A function that performs a specific task.


# TOOL CALLING
# ↓
# LLM decides:
# "Which tool should I call?"
# and provides arguments.


# AGENT
# ↓
# Manages the repeated process:
# think → choose tool → execute → observe result
# → choose again → final answer